# Caching

## 1. What is caching? 

Imagine our KYC app has to load a **huge textbook** every time someone asks a question.

Without caching:

> User 1 → open textbook → read it
> User 2 → open textbook again → read it
> User 3 → open textbook again → read it 😩

With caching:

> User 1 → open textbook → keep it ready
> User 2 → use the already-open textbook
> User 3 → use the same one

So:

> **Caching = remembering something expensive so we don't have to calculate/load it again.**

This makes the application **faster** and can also reduce **CPU/GPU usage and API calls**.

---

# 2. Why is caching important in our KYC app?

Our application has some expensive things.

For example:

### EasyOCR model

We have:

```python
easyocr.Reader(["en"], gpu=False)
```

Loading the OCR model takes time and memory.

We **don't** want to load the entire OCR model every time a user clicks Submit.

### dlib models

Same thing.

We have:

```python
dlib.shape_predictor(...)
dlib.face_recognition_model_v1(...)
```

These models are expensive to load.

### KYC policy

Our policy doesn't change every time a user submits KYC.

So there's no point reading/rebuilding it repeatedly.

### Policy embeddings

This is particularly important.

We send policy chunks to OpenAI to generate embeddings.

That means:

**API call → time + cost**

If the policy hasn't changed, we should reuse the embeddings.

---

# 3. Streamlit gives us two important caching types

You already have both in your code.

## A. `st.cache_resource`

Think:

> **"Keep this expensive resource alive and reuse it."**

Good for things like:

* ML models
* OCR models
* database connections
* clients/resources

You already use it:

```python
@st.cache_resource
def load_ocr_model():
    return easyocr.Reader(["en"], gpu=False)
```

And:

```python
@st.cache_resource
def load_dlib_models():
    ...
```

### What happens?

First time:

```text
User → load OCR model → expensive
```

Next time:

```text
User → reuse OCR model → much faster
```

So **this is correct**. ✅

---

# 4. `st.cache_data`

Think:

> **"Remember the result of a calculation/function."**

Good for things like:

* processed data
* calculations
* database query results
* API results
* static data

You already use it for:

```python
@st.cache_data
def load_kyc_policy():
```

and:

```python
@st.cache_data
def create_policy_embeddings(collection_chunks):
```

That's also correct. ✅

---

# 5. The important difference

Easy way to remember:

| Cache            | Think of it as      | Our example       |
| ---------------- | ------------------- | ----------------- |
| `cache_resource` | Keep the **thing**  | OCR/dlib models   |
| `cache_data`     | Keep the **result** | Policy/embeddings |

So:

**Model = resource**

**Model's calculated/output data = data**

---

# 6. What happens when the policy changes?

This is an important concept.

Suppose today our policy is:

> Face distance must be below 0.50.

The embeddings are generated and cached.

Tomorrow we change the policy.

The cached embeddings may no longer represent the new policy.

Streamlit's caching system uses the function's inputs to determine whether it can reuse the cached result.

So if our input changes, the cached result can be recomputed.

That's one reason this is useful:

```python
create_policy_embeddings(collection_chunks)
```

The `collection_chunks` are an input.

If they change, Streamlit can generate fresh embeddings.

---

# 7. Do we need MORE caching in our KYC app?

**No. Not right now.**

You already have the important caching:

### ✅ OCR model

```python
@st.cache_resource
```

### ✅ dlib models

```python
@st.cache_resource
```

### ✅ KYC policy

```python
@st.cache_data
```

### ✅ Policy embeddings

```python
@st.cache_data
```



# 8. But then what does "Performance Optimization" mean?

Caching is only **one part** of performance optimization.

We look at our latency results:

| Stage        |   Typical time |
| ------------ | -------------: |
| S3           |       ~4–7 sec |
| OCR          |     ~0.5–3 sec |
| Face         |         <1 sec |
| **Liveness** |  **~9–16 sec** |
| RAG          |     ~0.7–4 sec |
| **LLM**      | **~10–14 sec** |

We already discovered that:

**Liveness + LLM are our biggest recurring costs.**

Caching won't magically fix those.

For example, we **cannot cache liveness results** because every user's video is different.

We also shouldn't cache an LLM KYC decision because the evidence is different for every applicant.

So performance optimization means asking:

> **"What expensive work can we avoid repeating, reduce, or make faster without hurting correctness?"**




## What is Performance Optimization?

Your KYC system does many things one after another:

**Upload → Validate → OCR → Face → Liveness → RAG → LLM → Result**

Performance optimization means:

> **Make the same system do the same job faster, without making the result worse.**

We experimented with a few things.

---

## 1. Caching ✅ — already completed

This was our first optimization area.

### Problem

Some things don't need to be loaded again and again.

For example:

```python
easyocr.Reader(["en"], gpu=False)
```

and your dlib models are expensive to load.

### What we did

We used:

```python
@st.cache_resource
```

for the OCR and dlib models.

We also cached the KYC policy and policy embeddings where appropriate.

### Easy example

Without caching:

```text
Run 1 → load model
Run 2 → load model again
Run 3 → load model again
```

With caching:

```text
Run 1 → load model
Run 2 → reuse model
Run 3 → reuse model
```

### Result

✅ **Caching is DONE.**

We are not working on caching anymore.

---

# 2. LLM Prompt Optimization ❌ Tested, rejected

Your LLM takes roughly **10–14 seconds** in many runs.

So we thought:

> Maybe the prompt is too long and we can make it shorter.

Your original prompt had **15 rules**.

We tried reducing it.

### Experiment A — 9 rules

We removed some repeated instructions.

Result:

* Total time around **39 sec**
* You felt the answer quality was slightly worse.

So:

❌ **Rejected**

---

### Experiment B — 12 rules

We created a cleaner 12-rule version.

Result:

* Around **35 sec** in that particular run.
* Output was okay, but you preferred the original style.

And when we repeated the same files, the LLM took:

```text
13.917 sec
```

This showed that timing naturally fluctuates.

So there was **no reliable proof** that the shorter prompt was faster.

### Final decision

✅ **Keep the original 15-rule prompt.**

Why?

Because in KYC:

> **Correctness > tiny prompt reduction**

And our original output was clearer to you.

---

# 3. Output token limit ❌ Tested, rejected

We tried:

```python
max_completion_tokens=150
```

The idea was:

> "The answer is short, so tell the model not to generate a huge answer."

our expected output is only:

```text
Decision
Reason
Recommended Action
```

So 150 seemed enough.

### What happened?

The assessment still worked.

But your LLM time was:

```text
12.475 sec
```

So we did **not see a meaningful speed improvement**.

Therefore:

❌ We removed `max_completion_tokens=150`.

### Lesson

Just because an output is short doesn't mean the model will necessarily respond faster when you set a token limit.

---

# 4. RAG Top-K Optimization ❌ Tested, rejected

This one was more interesting.

Your RAG retrieves the most relevant policy chunks.

Initially:

```python
top_chunks = retrieval_results[:3]
```

Meaning:

> Take the best 3 policy chunks.

We thought:

> Maybe 2 chunks are enough.

So we changed only:

```python
top_chunks = retrieval_results[:2]
```

### What happened?

This was BAD for your KYC system.

Your evidence was:

```text
Face → PASS
Liveness → REVIEW
```

But the LLM returned:

```text
Decision: PASS
```

That is incorrect for your intended overall KYC decision.

The original policy explicitly says the liveness condition with blink > 0 and lip movement = 0 is REVIEW, and defines REVIEW as an overall inconclusive state.  

And it wasn't faster either:

```text
RAG   = 2.006 sec
LLM   = 21.120 sec
Total = 35.138 sec
```

So:

❌ **Top 2 rejected**

We restored:

```python
top_chunks = retrieval_results[:3]
```

### Important lesson

More context isn't always bad.

In your case:

**3 chunks → better policy understanding**

**2 chunks → lost important policy context**

---

# 5. What we learned overall

The biggest lesson from these experiments is:

> **Don't optimize something just because it looks theoretically expensive. Test it and see whether speed AND correctness actually improve.**

For your KYC project:

| Optimization   | What we tried                       | Result                           |
| -------------- | ----------------------------------- | -------------------------------- |
| Caching        | Cache OCR, dlib, policy, embeddings | ✅ Keep                           |
| Prompt         | 15 → 9 rules                        | ❌ Reject                         |
| Prompt         | 15 → 12 rules                       | ❌ Reject                         |
| Output limit   | 150 tokens                          | ❌ No useful speed gain           |
| RAG            | 3 → 2 chunks                        | ❌ Worse decision + no speed gain |
| Original setup | 15 rules + top 3 chunks             | ✅ Keep                           |

---

# 6. Why we haven't dramatically reduced total time

This is the most important part.

Look at one of your laptop runs:

```text
S3          ~7 sec
OCR         ~3 sec
Face        ~1 sec
Liveness    ~38 sec   ← BIG
RAG         ~4 sec
LLM         ~10 sec
```

So the biggest problem isn't your prompt.

It's:

### 🥇 Liveness/video processing

Especially with your longer laptop video.

Your phone runs were often around:

**28–37 seconds total**

while the larger laptop video runs were around:

**62–64 seconds total** in our earlier benchmark.

So if we really want a **big performance improvement**, we have to attack the video/liveness processing rather than keep tweaking wording.

---

# Where we stand now

```text
Caching
   ↓
✅ DONE

Prompt optimization
   ↓
✅ Tested
✅ Original 15 rules kept

RAG optimization
   ↓
✅ Tested
✅ Top 3 kept


